# F1 23 · Mônaco — Virtual Race Engineer
## Treino, comparação e *tuning* de modelos de Machine Learning

**Objetivo.** Prever o tempo de volta (s) em Mônaco a partir do *setup* do carro (Ferrari e Alfa Romeo, 11 setups S0–S10) e do contexto da volta (idade do pneu, combustível), comparando seis algoritmos clássicos contra um *baseline* e avaliando o efeito do ajuste de hiperparâmetros.

**Roteiro do notebook**
1. Instalação e importações
2. Carregamento de `voltas.csv` (upload, Google Drive, URL ou caminho local)
3. Preparação dos dados e definição das *features* (sem vazamento)
4. Análise exploratória rápida
5. Protocolo de validação (GroupKFold por sessão + hold-out de 20% das sessões)
6. Modelos com hiperparâmetros padrão
7. *Tuning* (RandomizedSearchCV aninhado)
8. Previsto × real, resíduos e importância das variáveis
9. Alvos secundários: velocidade máxima e velocidade média
10. Experimento: Ferrari S10 × Alfa S2
11. Salvamento dos artefatos e conclusões

> Dica: no menu **Ambiente de execução → Executar tudo**. Para um teste rápido, ajuste `N_ITER` na seção 1.

## 1. Instalação e importações

In [ ]:
%pip install -q scikit-learn pandas numpy matplotlib seaborn plotly joblib scipy

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
import joblib
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, RandomizedSearchCV, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SEED = 42          # reprodutibilidade
N_ITER = 30        # iterações do RandomizedSearchCV (reduza p/ 5 num teste rápido)
N_JOBS = -1
OUT = '/content/f1_monaco_out' if os.path.isdir('/content') else os.path.abspath('f1_monaco_out')
os.makedirs(OUT, exist_ok=True)
print('Artefatos em:', OUT)

## 2. Carregamento dos dados
Escolha **uma** das opções abaixo. A célula 2.4 procura o arquivo automaticamente nos caminhos conhecidos; as opções 2.1–2.3 só são necessárias no Colab.

### 2.1 Opção A — Upload do `voltas.csv` (Colab)
Descomente e execute; o arquivo vai para `/content/voltas.csv`.

In [ ]:
# from google.colab import files
# up = files.upload()   # selecione voltas.csv
# DATA_PATH = '/content/' + list(up.keys())[0]

### 2.2 Opção B — Google Drive
Monte o Drive e informe o caminho do arquivo.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/f1_monaco/voltas.csv'   # ajuste

### 2.3 Opção C — URL
Cole um link direto para o CSV (ex.: *raw* do GitHub).

In [ ]:
DATA_URL = ''   # ex.: 'https://raw.githubusercontent.com/<usuario>/<repo>/main/voltas.csv'
if DATA_URL:
    import urllib.request
    urllib.request.urlretrieve(DATA_URL, 'voltas.csv'); DATA_PATH = 'voltas.csv'

### 2.4 Localização automática e leitura

In [ ]:
cands = [globals().get('DATA_PATH'), '/content/voltas.csv', 'voltas.csv',
         '../relatorio/voltas.csv', '/workspace/f1_monaco/relatorio/voltas.csv']
DATA_PATH = next((p for p in cands if p and os.path.exists(p)), None)
assert DATA_PATH, 'voltas.csv não encontrado — use uma das opções 2.1–2.3'
raw = pd.read_csv(DATA_PATH)
print(DATA_PATH, raw.shape); raw.head()

## 3. Preparação dos dados e *features*
- Mantêm-se apenas **Ferrari** e **Alfa Romeo** (as poucas voltas de McLaren são descartadas) e voltas válidas (`valida == 1`).
- Alvo principal: `tempo_volta_s = tempo_volta_ms / 1000`.
- *Features*: parâmetros de setup + contexto conhecido **antes** da volta (idade do pneu, combustível, volta de saída/box) + *one-hot* de carro e composto.
- **Sem vazamento:** tempos de setor, velocidades mínimas/máximas, temperaturas e desgaste são *resultados* da volta e ficam fora do modelo de tempo.
- Colunas constantes (ex.: lastro, se não variou) são removidas automaticamente.

In [ ]:
d = raw[raw.carro.isin(['ferrari', 'alfaromeo'])].copy()
if 'valida' in d: d = d[d.valida == 1]
d['tempo_volta_s'] = d.tempo_volta_ms / 1000
d = d.reset_index(drop=True)

SETUP = ['asa_dianteira','asa_traseira','dif_com_acel','dif_sem_acel','cambagem_diant','cambagem_tras',
         'convergencia_diant','convergencia_tras','suspensao_diant','suspensao_tras','barra_diant','barra_tras',
         'altura_diant','altura_tras','freio_pressao','freio_balanco','freio_motor',
         'pneu_tras_esq_psi','pneu_tras_dir_psi','pneu_diant_esq_psi','pneu_diant_dir_psi','lastro','combustivel_setup']
CTX = ['idade_pneu_voltas','volta_saida','entrou_box']
cols = [c for c in SETUP + CTX if c in d and d[c].notna().any() and d[c].nunique() > 1]
X = d[cols].astype(float)
X = X.loc[:, ~X.T.duplicated()]          # remove colunas idênticas (ex.: pneus esq = dir)
X['carro_ferrari'] = (d.carro == 'ferrari').astype(int)
if 'composto_visual' in d and d.composto_visual.nunique() > 1:
    X = X.join(pd.get_dummies(d.composto_visual, prefix='composto', drop_first=True).astype(int))
X = X.fillna(X.median())
y = d.tempo_volta_s.values
groups = d.session_uid.values
FEATURES = list(X.columns)
print(f'{len(d)} voltas · {d.session_uid.nunique()} sessões · {len(FEATURES)} features')
print(FEATURES)

## 4. Análise exploratória rápida
### 4.1 Tempo de volta por setup e carro

In [ ]:
fig = px.box(d, x='setup_id', y='tempo_volta_s', color='carro', points='all',
             category_orders={'setup_id': sorted(d.setup_id.unique(), key=lambda s: int(s[1:]))},
             title='Tempo de volta por setup × carro', labels={'tempo_volta_s':'Tempo (s)'})
fig.show()
print(d.groupby(['setup_id','carro']).tempo_volta_s.agg(['count','mean','min']).round(3).unstack())

### 4.2 Degradação: tempo médio × idade do pneu

In [ ]:
g = d.groupby(['idade_pneu_voltas','carro']).tempo_volta_s.mean().reset_index()
plt.figure(figsize=(8,4)); sns.lineplot(data=g, x='idade_pneu_voltas', y='tempo_volta_s', hue='carro', marker='o')
plt.title('Tempo médio por idade do pneu'); plt.ylabel('Tempo (s)'); plt.tight_layout()
plt.savefig(f'{OUT}/tempo_vs_idade_pneu.png', dpi=120); plt.show()

## 5. Protocolo de validação
Voltas da mesma sessão são correlacionadas; por isso todas as divisões são **por sessão** (`session_uid`):
- **Validação cruzada:** `GroupKFold(5)` sobre todo o conjunto.
- **Hold-out:** `GroupShuffleSplit` com 20% das sessões para teste (seed 42).
- **Tuning aninhado:** a busca roda com `GroupKFold(4)` *apenas no treino* de cada dobra externa — a métrica tunada é honesta.

In [ ]:
gkf = GroupKFold(n_splits=5)
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(X, y, groups))
Xtr, Xte, ytr, yte, gtr = X.iloc[tr], X.iloc[te], y[tr], y[te], groups[tr]
print(f'Treino: {len(tr)} voltas / {len(set(gtr))} sessões · Teste: {len(te)} voltas / {len(set(groups[te]))} sessões')

def metricas(yt, p):
    return dict(MAE=mean_absolute_error(yt, p), RMSE=mean_squared_error(yt, p)**0.5, R2=r2_score(yt, p))

## 6. Modelos com hiperparâmetros padrão
Lineares regularizados (Lasso, Ridge, ElasticNet) com padronização; ensembles de árvores (Random Forest, Extra Trees, Gradient Boosting). O *baseline* prevê sempre a média do treino.

In [ ]:
S = lambda m: make_pipeline(StandardScaler(), m)
ARV = {'n_estimators': randint(200, 600), 'max_depth': [3, 5, 8, None],
       'min_samples_leaf': randint(1, 20), 'max_features': [.3, .5, .8, 1.0]}
MODELOS = {   # nome: (estimador padrão, espaço de busca) — mesmos espaços de hiperparametros/tuning.py
 'Lasso':      (S(Lasso(max_iter=50000)), {'lasso__alpha': loguniform(1e-3, 1)}),
 'Ridge':      (S(Ridge()), {'ridge__alpha': loguniform(1e-2, 1e4)}),
 'ElasticNet': (S(ElasticNet(max_iter=50000)), {'elasticnet__alpha': loguniform(1e-3, 2), 'elasticnet__l1_ratio': uniform(.05, .95)}),
 'Random Forest':  (RandomForestRegressor(random_state=SEED, n_jobs=1), ARV),
 'Extra Trees':    (ExtraTreesRegressor(random_state=SEED, n_jobs=1), ARV),
 'Gradient Boosting': (GradientBoostingRegressor(random_state=SEED),
     {'n_estimators': randint(50, 500), 'max_depth': randint(1, 5), 'learning_rate': loguniform(.005, .2),
      'subsample': uniform(.5, .5), 'min_samples_leaf': randint(1, 20)}),
}

def avaliar(nome, est, versao, Xa=X, ya=y):
    p_cv = cross_val_predict(est, Xa, ya, cv=gkf, groups=groups)
    p_te = est.fit(Xa.iloc[tr], ya[tr]).predict(Xa.iloc[te])
    r = {'modelo': nome, 'versao': versao}
    r.update({f'{k}_CV': v for k, v in metricas(ya, p_cv).items()})
    r.update({f'{k}_teste': v for k, v in metricas(ya[te], p_te).items()})
    return r, p_cv, p_te

linhas, PRED_CV, PRED_TE = [], {}, {}
r, PRED_CV['Baseline'], PRED_TE['Baseline'] = avaliar('Baseline (média)', DummyRegressor(), '-')
linhas.append(r)
for nome, (est, _) in MODELOS.items():
    r, PRED_CV[nome], PRED_TE[nome] = avaliar(nome, est, 'padrão'); linhas.append(r)
    print(f"{nome:18s} MAE CV={r['MAE_CV']:.3f}s  teste={r['MAE_teste']:.3f}s")
pd.DataFrame(linhas).round(3)

## 7. *Tuning* de hiperparâmetros (RandomizedSearchCV aninhado)
Para cada modelo: (i) CV externa de 5 dobras em que a busca (30 combinações, `GroupKFold(4)`) usa só o treino da dobra; (ii) busca no treino do hold-out e avaliação no teste; (iii) busca final em todos os dados para registrar os melhores parâmetros.
⏱️ Leva alguns minutos no Colab (≈5–15 min com `N_ITER=30`).

In [ ]:
def busca(est, esp):
    return RandomizedSearchCV(est, esp, n_iter=N_ITER, cv=GroupKFold(4), scoring='neg_mean_absolute_error',
                              random_state=SEED, n_jobs=N_JOBS)
MELHORES, TUNADOS = {}, {}
for nome, (est, esp) in MODELOS.items():
    p_cv = np.zeros(len(y))
    for a, b in gkf.split(X, y, groups):
        p_cv[b] = busca(est, esp).fit(X.iloc[a], y[a], groups=groups[a]).predict(X.iloc[b])
    s_tr = busca(est, esp).fit(Xtr, ytr, groups=gtr); p_te = s_tr.predict(Xte)
    s_all = busca(est, esp).fit(X, y, groups=groups)
    MELHORES[nome] = {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in s_all.best_params_.items()}
    TUNADOS[nome] = s_tr.best_estimator_          # treinado só no treino (usado nos gráficos do teste)
    PRED_CV[nome + ' (tunado)'], PRED_TE[nome + ' (tunado)'] = p_cv, p_te
    r = {'modelo': nome, 'versao': 'tunado'}
    r.update({f'{k}_CV': v for k, v in metricas(y, p_cv).items()})
    r.update({f'{k}_teste': v for k, v in metricas(yte, p_te).items()})
    linhas.append(r); print(f"{nome:18s} MAE CV tunado={r['MAE_CV']:.3f}s  teste={r['MAE_teste']:.3f}s")
RES = pd.DataFrame(linhas)

### 7.1 Tabela: MAE padrão × tunado e melhores parâmetros

In [ ]:
cmp = RES[RES.versao != '-'].pivot(index='modelo', columns='versao', values=['MAE_CV', 'MAE_teste'])
cmp.columns = [f'{a}_{b}' for a, b in cmp.columns]
cmp['ganho_CV_%'] = 100 * (cmp['MAE_CV_padrão'] - cmp['MAE_CV_tunado']) / cmp['MAE_CV_padrão']
cmp = cmp.sort_values('MAE_CV_tunado')
base = RES[RES.versao == '-'].iloc[0]
print(f"Baseline: MAE CV={base.MAE_CV:.3f}s · teste={base.MAE_teste:.3f}s")
display(cmp.round(3))
display(pd.DataFrame({k: [json.dumps(v, default=str)] for k, v in MELHORES.items()}).T.rename(columns={0: 'melhores parâmetros'}))

### 7.2 Gráfico: MAE (CV) antes e depois do tuning

In [ ]:
pl = RES[RES.versao != '-'].copy()
fig = px.bar(pl, x='modelo', y='MAE_CV', color='versao', barmode='group', text_auto='.3f',
             title='MAE na validação cruzada (s) — padrão × tunado')
fig.add_hline(y=base.MAE_CV, line_dash='dash', annotation_text='Baseline (média)'); fig.show()
plt.figure(figsize=(9,4)); sns.barplot(data=pl, x='modelo', y='MAE_CV', hue='versao')
plt.axhline(base.MAE_CV, ls='--', c='k', label='baseline'); plt.xticks(rotation=20); plt.ylabel('MAE CV (s)')
plt.legend(); plt.tight_layout(); plt.savefig(f'{OUT}/mae_padrao_vs_tunado.png', dpi=120); plt.show()

## 8. Melhor modelo: previsto × real e resíduos
Escolhe-se o modelo de menor **MAE de CV** (honesta, aninhada). Os gráficos usam as previsões fora-da-amostra da CV (todas as voltas) e a tabela de exemplos usa o hold-out.

In [ ]:
melhor = RES[RES.versao != '-'].sort_values('MAE_CV').iloc[0]
CHAVE = melhor.modelo + (' (tunado)' if melhor.versao == 'tunado' else '')
print('Melhor modelo:', CHAVE, f'— MAE CV {melhor.MAE_CV:.3f}s · teste {melhor.MAE_teste:.3f}s')
pcv = PRED_CV[CHAVE]; res = y - pcv
dd = d[['carro','setup_id','idade_pneu_voltas']].assign(real=y, previsto=pcv, residuo=res)
fig = px.scatter(dd, x='real', y='previsto', color='carro', hover_data=['setup_id','idade_pneu_voltas'],
                 title=f'Previsto × real (CV) — {CHAVE}')
lo, hi = y.min(), y.max(); fig.add_shape(type='line', x0=lo, y0=lo, x1=hi, y1=hi, line=dict(dash='dash')); fig.show()

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
sns.scatterplot(data=dd, x='real', y='previsto', hue='carro', ax=ax[0]); ax[0].plot([lo, hi], [lo, hi], 'k--'); ax[0].set_title('Previsto × real')
sns.scatterplot(data=dd, x='previsto', y='residuo', hue='carro', ax=ax[1]); ax[1].axhline(0, c='k', ls='--'); ax[1].set_title('Resíduos × previsto')
sns.histplot(res, kde=True, ax=ax[2]); ax[2].set_title(f'Histograma dos resíduos (média {res.mean():.2f}s)')
plt.tight_layout(); plt.savefig(f'{OUT}/previsto_real_residuos.png', dpi=120); plt.show()

### 8.1 Exemplos do hold-out: real × previsto × erro

In [ ]:
amostra = d.iloc[te][['carro','setup_id','volta','idade_pneu_voltas','tempo_volta_s']].copy()
amostra['previsto_s'] = PRED_TE[CHAVE]; amostra['erro_s'] = amostra.previsto_s - amostra.tempo_volta_s
amostra.sample(min(15, len(amostra)), random_state=SEED).sort_values(['carro','setup_id']).round(3)

### 8.2 Importância das variáveis
Importância por **permutação** (aumento do MAE ao embaralhar a variável), no conjunto de teste, para o melhor modelo tunado treinado no treino. Para lineares, mostra-se também |coeficiente| padronizado.

In [ ]:
est_imp = TUNADOS[melhor.modelo]
pi = permutation_importance(est_imp, Xte, yte, n_repeats=20, random_state=SEED, scoring='neg_mean_absolute_error')
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False)
top = imp.head(15)[::-1]
plt.figure(figsize=(8, 5)); top.plot.barh(color='firebrick'); plt.xlabel('Aumento do MAE (s)')
plt.title(f'Importância por permutação — {melhor.modelo} tunado'); plt.tight_layout()
plt.savefig(f'{OUT}/importancia_features.png', dpi=120); plt.show()
print((100 * imp.clip(lower=0) / imp.clip(lower=0).sum()).head(5).round(1).rename('% da importância'))
ult = est_imp[-1] if hasattr(est_imp, 'steps') else est_imp
if hasattr(ult, 'coef_'):
    display(pd.Series(ult.coef_, index=FEATURES).sort_values(key=abs, ascending=False).head(10).round(3).rename('coef padronizado (s/DP)'))

## 9. Alvos secundários: velocidade máxima e velocidade média
Mesmas *features* e protocolo (modelos padrão + baseline). Espera-se que a **velocidade máxima** dependa pouco do setup (pico no túnel) e a **velocidade média** responda bem a ele.

In [ ]:
sec = []
for alvo in ['vel_max_kmh', 'vel_media_kmh']:
    if alvo not in d: continue
    ya = d[alvo].values.astype(float)
    for nome, est in [('Baseline (média)', DummyRegressor())] + [(n, e) for n, (e, _) in MODELOS.items()]:
        r, _, _ = avaliar(nome, est, 'padrão', X, ya); r['alvo'] = alvo; sec.append(r)
SEC = pd.DataFrame(sec)
display(SEC.pivot(index='modelo', columns='alvo', values='MAE_CV').round(2))
g = sns.catplot(data=SEC, x='modelo', y='MAE_CV', col='alvo', kind='bar', sharey=False, height=4, aspect=1.4, color='steelblue')
for a in g.axes.flat: a.tick_params(axis='x', rotation=30)
g.set_axis_labels('', 'MAE CV (km/h)'); plt.tight_layout(); plt.savefig(f'{OUT}/mae_velocidades.png', dpi=120); plt.show()

## 10. Experimento: Ferrari S10 × Alfa S2
Monta-se uma volta “típica” de cada configuração (mediana das *features* daquele carro+setup) e compara-se a previsão do melhor modelo (re-treinado em todos os dados) com a média real observada. Também se mostra o S0 como referência.

In [ ]:
final = TUNADOS[melhor.modelo] if melhor.versao == 'tunado' else MODELOS[melhor.modelo][0]
from sklearn.base import clone
final = clone(final).fit(X, y)
exp = []
for carro, setup in [('ferrari','S0'), ('ferrari','S10'), ('alfaromeo','S0'), ('alfaromeo','S2'), ('alfaromeo','S10')]:
    m = (d.carro == carro) & (d.setup_id == setup)
    if not m.any(): continue
    linha = X[m].median().to_frame().T
    exp.append(dict(carro=carro, setup=setup, voltas=int(m.sum()), real_medio_s=d.tempo_volta_s[m].mean(),
                    melhor_real_s=d.tempo_volta_s[m].min(), previsto_s=float(final.predict(linha)[0])))
EXP = pd.DataFrame(exp); EXP['erro_s'] = EXP.previsto_s - EXP.real_medio_s
fmt = lambda s: f'{int(s//60)}:{s%60:06.3f}'
display(EXP.assign(real=EXP.real_medio_s.map(fmt), previsto=EXP.previsto_s.map(fmt), melhor=EXP.melhor_real_s.map(fmt)).round(3))

### 10.1 Simulador: altere o setup e veja a previsão
Edite os valores do dicionário `ajustes` (ex.: `dif_sem_acel`) partindo da mediana de um setup.

In [ ]:
base_m = (d.carro == 'ferrari') & (d.setup_id == 'S10')
linha = X[base_m].median().to_frame().T
ajustes = {'dif_sem_acel': 80}          # ← mude aqui
for k, v in ajustes.items():
    if k in linha: linha[k] = v
print(f'Previsto: {fmt(final.predict(linha)[0])} (base Ferrari S10 com {ajustes})')

## 11. Salvamento dos artefatos

In [ ]:
RES.round(4).to_csv(f'{OUT}/metricas_tempo_volta.csv', index=False)
cmp.round(4).to_csv(f'{OUT}/mae_padrao_vs_tunado.csv')
SEC.round(4).to_csv(f'{OUT}/metricas_velocidades.csv', index=False)
EXP.round(4).to_csv(f'{OUT}/experimento_setups.csv', index=False)
json.dump(MELHORES, open(f'{OUT}/melhores_parametros.json', 'w'), indent=1, default=str)
joblib.dump({'modelo': final, 'features': FEATURES}, f'{OUT}/melhor_modelo.joblib')
print(sorted(os.listdir(OUT)))
# No Colab, para baixar tudo:  !zip -r /content/f1_monaco_out.zip /content/f1_monaco_out
# from google.colab import files; files.download('/content/f1_monaco_out.zip')

## 12. Conclusões (calculadas a partir desta execução)

In [ ]:
t = cmp.sort_values('MAE_CV_tunado')
print(f"• Baseline (média): MAE CV {base.MAE_CV:.3f}s")
pad = RES[RES.versao == 'padrão'].sort_values('MAE_CV').iloc[0]
print(f"• Antes do tuning, o melhor foi {pad.modelo} (MAE CV {pad.MAE_CV:.3f}s).")
print(f"• Depois do tuning: 1º {t.index[0]} ({t.MAE_CV_tunado.iloc[0]:.3f}s), 2º {t.index[1]} ({t.MAE_CV_tunado.iloc[1]:.3f}s).")
print(f"• Maior ganho do tuning: {t['ganho_CV_%'].idxmax()} ({t['ganho_CV_%'].max():.1f}%).")
print(f"• Variáveis mais importantes ({melhor.modelo}): {', '.join(imp.index[:3])}.")
for alvo in SEC.alvo.unique():
    s = SEC[SEC.alvo == alvo].set_index('modelo').MAE_CV; b = s['Baseline (média)']; s2 = s.drop('Baseline (média)')
    print(f"• {alvo}: melhor {s2.idxmin()} ({s2.min():.2f} km/h) vs baseline {b:.2f} km/h ({100*(b-s2.min())/b:.0f}% melhor).")
print('Leitura: se lineares regularizados lideram, a relação setup→tempo é quase aditiva; pneu e diferencial dominam.')

---
### (Opcional) Rede neural MLP
Fora do escopo principal: com ~270 voltas a MLP tende a sobreajustar. Descomente para testar.

In [ ]:
# from sklearn.neural_network import MLPRegressor
# mlp = S(MLPRegressor(hidden_layer_sizes=(16,), alpha=1.0, max_iter=3000, random_state=SEED))
# r, _, _ = avaliar('MLP', mlp, 'padrão'); print(r)